# Credit card offers — clean Colab workflow

Use a **fresh Colab runtime** and run the eight code cells in order.
Upload the **cleaned project ZIP**, not the old patch project or an Excel file.
The project already contains your unchanged 257-row CSV.

This is an assignment research demo. The dataset contains source-linked candidates,
some eligibility details remain unverified, and Low-class evaluation has only three
test examples. Saved model probabilities are estimates.


## 1. Upload and extract the cleaned project
If you already ran this cell in the same session, it reuses the extracted folder.
It does not overwrite the old project folder or rewrite Python files.


In [ ]:
from pathlib import Path
import io
import stat
import zipfile

ROOT = Path('/content/credit_card_offers_cleaned')
PROJECT = ROOT / 'cc_offers_app'

if not (PROJECT / 'train.py').exists():
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Choose exactly one cleaned project ZIP.')
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        for member in archive.infolist():
            target = (ROOT / member.filename).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Archive contains an unsafe path.')
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Archive contains a symbolic link.')
        archive.extractall(ROOT)

if not (PROJECT / 'train.py').exists():
    raise ValueError('This is not the cleaned project ZIP. Upload the supplied cleaned ZIP.')
print('Project ready:', PROJECT)


## 2. Install the project packages
Pinned versions match the supplied experiment. Installation can take a few minutes.
If Colab requests a runtime restart, restart, then rerun from step 1.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')],
    check=True
)
print('Packages installed.')


## 3. Check the dataset and group split
This does not train models. All related rows remain together, including the reserved
demo group. Missing card tiers or networks remain unknown.


In [ ]:
check_script = """
from data import load_data, partition_data, TARGET
frame = load_data()
parts, demo, manifest, folds = partition_data(frame)
print('Rows:', len(frame))
print('Classes:')
print(frame[TARGET].value_counts().to_string())
for name, part in parts.items():
    print(name, len(part['X']), 'rows;', part['groups'].nunique(), 'groups')
print('Demo rows:', int((manifest['Partition'] == 'demo').sum()))
assert manifest.groupby('Split Group')['Partition'].nunique().max() == 1
print('Grouped CV folds:', len(folds))
print('Data and split checks passed.')
"""
subprocess.run([sys.executable, '-u', '-c', check_script], cwd=PROJECT, check=True)


## 4. Load the compatible model or train once
The supplied ZIP includes a checked model. This cell reuses it when the data, training
code and scikit-learn version match. Otherwise it trains and regenerates results.
Set `TRAIN_AGAIN = True` only when you intend to repeat the experiment.
Training may take several minutes; do not start another training cell concurrently.


In [ ]:
TRAIN_AGAIN = False

training_script = """
import json
import sys
from pathlib import Path
from data import dataset_hash
from run_metadata import cache_is_current
from train import train_all

root = Path.cwd()
metadata_path = root / 'model_metadata.json'
compatible = False
if metadata_path.exists() and (root / 'models.joblib').exists():
    metadata = json.loads(metadata_path.read_text())
    compatible = cache_is_current(metadata, dataset_hash())

if compatible and sys.argv[1] != 'True':
    print('Using the compatible saved model:', metadata['best'])
    print('Generated:', metadata['generated_at_utc'])
else:
    train_all(overwrite=True)
"""
subprocess.run(
    [sys.executable, '-u', '-c', training_script, str(TRAIN_AGAIN)],
    cwd=PROJECT, check=True
)


## 5. Run the checks
Checks cover group separation, target leakage, API validation, predictions with unseen
inputs, SHAP additivity, LIME diagnostics, and rejection of stale models.


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT, check=True
)


## 6. View current results and explanations
`results/` is the regenerated run. `reference_results/` preserves the run you originally
uploaded. Use one run consistently in your report. SHAP/LIME below explain a saved
validation example, not arbitrary inputs entered later in the app.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

results = PROJECT / 'results'
metadata = json.loads((PROJECT / 'model_metadata.json').read_text())
slug = metadata['best'].lower().replace(' ', '_')

print('Selected by validation macro F1:', metadata['best'])
print('Dataset rows:', metadata['dataset_rows'])
print('Partition counts:', metadata['partition_counts'])
display(pd.read_csv(results / 'model_comparison.csv'))
print((results / slug / 'classification_report.txt').read_text())
print('LIME local fit diagnostics:')
print((results / 'lime' / f'{slug}_summary.txt').read_text())

for relative_path in [
    f'{slug}/confusion_matrix.png',
    f'{slug}/feature_importance.png',
    f'shap/{slug}_global.png',
    f'shap/{slug}_local.png',
    f'lime/{slug}_explanation.png',
]:
    display(Image(filename=str(results / relative_path)))


## 7. Open the app
Keep the runtime connected. Click **Load reserved demo sample**, then **Predict
Discount Tier**. Edit fields to try another input. The app reports unseen inputs;
probabilities do not confirm actual merchant eligibility.


In [ ]:
import time
import urllib.request
import socket

PORT = 5000
if 'app_process' in globals() and app_process.poll() is None:
    app_process.terminate()
    try:
        app_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        app_process.kill()
        app_process.wait()

# Avoid displaying an older app already listening on this port.
try:
    with socket.socket() as probe:
        probe.bind(('127.0.0.1', PORT))
except OSError as error:
    raise RuntimeError('Port 5000 is already in use. Use a fresh runtime or change PORT above.') from error

log_path = ROOT / 'app.log'
with log_path.open('w') as log:
    app_process = subprocess.Popen(
        [sys.executable, '-u', 'app.py', '--port', str(PORT)],
        cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT
    )

ready = False
for _ in range(30):
    if app_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=1) as response:
            ready = response.status == 200
        if ready:
            break
    except Exception:
        time.sleep(0.5)

if not ready:
    print(log_path.read_text())
    raise RuntimeError('App did not become ready. Check the log above. Do not retrain blindly.')

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=1100)


## 8. Download the project
This includes the real CSV, clean source, notebook, current models/results and original
reference results. It excludes temporary files and bytecode. Also save this notebook
using **File → Download → Download .ipynb** if you added notes of your own.


In [ ]:
from google.colab import files

zip_path = ROOT / 'cc_offers_cleaned.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PROJECT.rglob('*')):
        if not path.is_file():
            continue
        if any(part in {'__pycache__', '.venv', 'venv'} or part.startswith('.training-') for part in path.parts):
            continue
        if path.suffix in {'.pyc', '.log'}:
            continue
        archive.write(path, path.relative_to(PROJECT.parent))

files.download(str(zip_path))
